In [1]:
%reload_ext autoreload
%autoreload 2

from scipy.constants import mu_0
import MaNTA
from Objective import make_objective


import jax
import jax.numpy as jnp
import equinox as eqx
import yancc

from yancc_wrapper import yancc_data

import desc
from desc import set_device
set_device("gpu")
import desc.io
from desc.equilibrium import Equilibrium
from desc.geometry import FourierRZToroidalSurface
from desc.grid import Grid, LinearGrid
from desc.objectives import (
    AspectRatio,
    FixBoundaryR,
    FixBoundaryZ,
    FixCurrent,
    FixPsi,
    ForceBalance,
    LinearObjectiveFromUser,
    ObjectiveFunction,
    ObjectiveFromUser,
    RotationalTransform,
    Volume,
)
from desc.profiles import SplineProfile



Using gpu implementation
[CudaDevice(id=0), CudaDevice(id=1), CudaDevice(id=2), CudaDevice(id=3)]


In [3]:
from Stellarator import StellaratorTransport

st_config = {
    "SourceCenter": 0.2,
    "SourceHeight": 250.0,
    "SourceWidth": 0.2,
    "EdgeTemperature":0.1,
    "EdgeDensity": 0.0,
    "n0": 0.5,
}
# runner = MaNTA.Runner(st)

# # %%
solver_config = {
    "OutputFilename": "stellarator_opt",
    "Polynomial_degree": 3,
    "Grid_size": 4,
    "tau": 100.0, 
    "Lower_boundary": 0.0,
    "Upper_boundary": 0.9,
    "Relative_tolerance": 0.01,
    "delta_t": 0.0001,
    "MinStepSize": 1e-8,
    "restart": False,
    "solveAdjoint": True, 
}

config = {
    "Stellarator": st_config,
    "Solver": solver_config,
}
points =  MaNTA.getNodes(solver_config["Lower_boundary"], solver_config["Upper_boundary"], solver_config["Grid_size"], solver_config["Polynomial_degree"])

yancc_rho = jnp.array(points)
yancc_ntheta = 17
yancc_nzeta = 33

# to allow maximum flexibility to match manta, we use a spline with the same control points as manta \
# + axis and lcfs
# initial pressure is all zeros, can change this if desired
pressure_rho = jnp.concatenate([jnp.zeros(1), yancc_rho, jnp.ones(1)])
desc_pressure = SplineProfile(jnp.zeros_like(pressure_rho), pressure_rho)

eq_est = desc.examples.get("ESTELL")
surf = eq_est.get_surface_at(rho=1)
eq = Equilibrium(M=4, N=4, Psi=0.1, surface=surf, pressure=desc_pressure)
eq = eq.solve(x_scale="ess")[0]

# eq = desc.io.load("eq_self_consistent_pressure.h5")
# desc_pressure = eq.get_profile('p')
eq_init = eq.copy()

V0 = eq.compute("V")["V"]
# yancc_wrapper = yancc_data.from_eq(points, grid = yancc_grid,rho = yancc_rho, Density=Density, eq=eq_init, nt = yancc_ntheta, nz = yancc_nzeta)
yancc_wrapper = yancc_data.from_eq(points, eq=eq_init)

st = StellaratorTransport(config, yancc_wrapper=yancc_wrapper)
st.run()


Building objective: force
Precomputing transforms
Building objective: lcfs R
Building objective: lcfs Z
Building objective: fixed Psi
Building objective: fixed pressure
Building objective: fixed current
Building objective: fixed sheet current
Building objective: self_consistency R
Building objective: self_consistency Z
Building objective: lambda gauge
Building objective: axis R self consistency
Building objective: axis Z self consistency
Number of parameters: 120
Number of objectives: 850

Starting optimization
Using method: lsq-exact
Optimization terminated successfully.
`ftol` condition satisfied. (ftol=1.00e-02)
         Current function value: 2.817e-05
         Total delta_x: 4.630e-01
         Iterations: 7
         Function evaluations: 8
         Jacobian evaluations: 8
                                                                 Start  -->   End
Total (sum of squares):                                      2.754e+00  -->   2.817e-05, 
Maximum absolute Force error:          

INFO: Using default value for configuration option Grid_points
INFO: Using default value for configuration option High_Grid_Boundary
INFO: Using default value for configuration option Lower_Boundary_Fraction
INFO: Using default value for configuration option Upper_Boundary_Fraction
INFO: Creating grid with 4 cells from x = 0 to x = 0.9
Total HDG degrees of freedom 53
INFO: Using default value for configuration option Absolute_tolerance
INFO: Using default value for configuration option tZero
INFO: Using default value for configuration option OutputPoints
INFO: Using default value for configuration option SteadyStateTolerance
INFO: Using default value for configuration option useCalcIC
INFO: Using default value for configuration option WriteOutput
Configuration done.
Setting initial conditions


before return
[-1.59917992e-03 -3.59527123e-02 -1.93881130e-01 -4.26012758e-01
 -5.14956373e-01 -9.31224580e-01 -1.84906604e+00 -2.79643200e+00
 -3.11746811e+00 -4.47310687e+00 -6.99514584e+00 -9.31856560e+00
 -1.00528343e+01 -1.29922409e+01 -1.78981426e+01 -2.17752536e+01]
before return
[3.49679309e+02 3.29522668e+02 2.58650645e+02 1.94878046e+02
 1.76979515e+02 1.18414006e+02 5.72648174e+01 3.06338755e+01
 2.52651634e+01 1.20023668e+01 3.57608837e+00 1.35827284e+00
 1.01734269e+00 3.43144282e-01 6.29904413e-02 1.69870567e-02]


Residual norm at t = 0: 0


before return
[-1.59917992e-03 -3.59527123e-02 -1.93881130e-01 -4.26012758e-01
 -5.14956373e-01 -9.31224580e-01 -1.84906604e+00 -2.79643200e+00
 -3.11746811e+00 -4.47310687e+00 -6.99514584e+00 -9.31856560e+00
 -1.00528343e+01 -1.29922409e+01 -1.78981426e+01 -2.17752536e+01]
before return
[3.49679309e+02 3.29522668e+02 2.58650645e+02 1.94878046e+02
 1.76979515e+02 1.18414006e+02 5.72648174e+01 3.06338755e+01
 2.52651634e+01 1.20023668e+01 3.57608837e+00 1.35827284e+00
 1.01734269e+00 3.43144282e-01 6.29904413e-02 1.69870567e-02]


...

Residual norm at t = 0.000187029: 2.53826


before return
[-33786.45113022  -7803.58096249  -1282.36134375   -606.8752872
     34.96317044   -102.99936157    -80.23973386    -71.14927929
    -75.52643378    -61.12001958    -50.95790929    -46.0340735
    -44.7343588     -41.76706106    -39.34350121    -38.80526866]
before return
[3.49679309e+02 3.29522668e+02 2.58650645e+02 1.94878046e+02
 1.76979515e+02 1.18414006e+02 5.72648174e+01 3.06338755e+01
 2.52651634e+01 1.20023668e+01 3.57608837e+00 1.35827284e+00
 1.01734269e+00 3.43144282e-01 6.29904413e-02 1.69870567e-02]


Residual norm at t = 0.000186859: 0.0256836


before return
[-2.86774652e+04 -6.78971595e+03 -1.18366936e+03 -5.49053463e+02
 -3.71879037e+00 -1.03873865e+02 -8.16132470e+01 -7.12418753e+01
 -7.37567717e+01 -6.08641446e+01 -5.07954233e+01 -4.59341460e+01
 -4.47048036e+01 -4.17057778e+01 -3.92929557e+01 -3.87543115e+01]
before return
[3.49679309e+02 3.29522668e+02 2.58650645e+02 1.94878046e+02
 1.76979515e+02 1.18414006e+02 5.72648174e+01 3.06338755e+01
 2.52651634e+01 1.20023668e+01 3.57608837e+00 1.35827284e+00
 1.01734269e+00 3.43144282e-01 6.29904413e-02 1.69870567e-02]


jax.debug.callback failed
Traceback (most recent call last):
  File "/global/homes/e/eatocco/.conda/envs/desc-env/lib/python3.12/runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/global/homes/e/eatocco/.conda/envs/desc-env/lib/python3.12/runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "/global/homes/e/eatocco/.conda/envs/desc-env/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/global/homes/e/eatocco/.conda/envs/desc-env/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/global/homes/e/eatocco/.conda/envs/desc-env/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 758, in start
    self.io_loop.start()
  File "/global/homes/e/eatocco/.conda/envs/desc-env/lib/python3.12/site-packages/tornado/platform/asyncio.py", line 211, in start
    self.asyncio_loop.run_forever()
  File "/globa

In [ ]:
solver_config = {
    "OutputFilename": "stellarator_opt",
    "Polynomial_degree": 3,
    "Grid_size": 4,
    "tau": 100.0, 
    "Lower_boundary": 0.0,
    "Upper_boundary": 0.9,
    "Relative_tolerance": 0.01,
    "delta_t": 1e-6,
    "MinStepSize": 1e-8,
    "useCalcIC": False,
    "restart": True,
    "solveAdjoint": True, 
}

config = {
    "Stellarator": st_config,
    "Solver": solver_config,
}

manta_objective = make_objective(config, vectorized=True)

# def manta_yancc_fun(fields, grid, Vprime):

#     stored_energy, pressure = Objective(fields, grid, Vprime) 

#     return stored_energy, pressure

def objective_from_user_fun(grid, data):
  # note: don't change the signature to this function
    yancc_dat = {
        "B_sup_t": data["B^theta"],
        "B_sup_z": data["B^zeta"],
        "B_sub_t": data["B_theta"],
        "B_sub_z": data["B_zeta"],
        "Bmag": data["|B|"],
        "dBdt": data["|B|_t"],
        "dBdz": data["|B|_z"],
        "sqrtg": data["sqrt(g)"],
    }

    yancc_dat = {
        key: grid.meshgrid_reshape(val, "rtz") for key, val in yancc_dat.items()
    }

    yancc_dat["Psi"] = grid.compress(
        data["Psi"] / grid.nodes[:, 0] ** 2, surface_label="rho"
    )
    yancc_dat["a_minor"] = jnp.full(grid.num_rho, data["a"])
    yancc_dat["R_major"] = jnp.full(grid.num_rho, data["R0"])
    yancc_dat["iota"] = grid.compress(data["iota"], surface_label="rho")
    yancc_dat["rho"] = grid.compress(grid.nodes[:, 0], surface_label="rho")

    V = grid.compress(data['V(r)'])
    V_r = grid.compress(data['V_r(r)'])
    Vprime = V_r/V[-1]

    fields = jax.vmap(lambda d: yancc.field.Field(**d, NFP=grid.NFP))(yancc_dat)

    desc_pressure = grid.compress(data["p"], surface_label="rho")
    
    stored_energy, manta_pressure = manta_objective(fields, grid, Vprime)
    
    
    pressure_error = manta_pressure - desc_pressure

    # optimization is easiest for least squares objectives, so instead of maximizing
    # stored energy we minimize 1/stored_energy^2 (the squaring happens later)
    return jnp.append(pressure_error, 1 / stored_energy)

    

In [ ]:
import interpax

yancc_ntheta = 17
yancc_nzeta = 33
yancc_rho = yancc_wrapper.rho
pressure_rho = jnp.concatenate([jnp.zeros(1), yancc_rho, jnp.ones(1)])
desc_pressure = SplineProfile(jnp.zeros_like(pressure_rho), pressure_rho)

# grid where desc needs to evaluate field for yancc/manta
yancc_desc_grid = LinearGrid(
    rho=yancc_rho, theta=yancc_ntheta, zeta=yancc_nzeta, NFP=eq.NFP
)


desc_data = eq.compute(["V(r)", "V_r(r)"], grid=yancc_desc_grid)

V = yancc_desc_grid.compress(desc_data['V(r)'])
Vn = V/V[-1] # normalize
Vn = interpax.CubicSpline(yancc_rho, Vn)

rho_from_normalized_volume = lambda Vnorm : desc.backend.root_scalar(lambda x: Vn(x) - Vnorm, jnp.sqrt(Vnorm))    

In [ ]:
domain_boundary_rho = rho_from_normalized_volume(0.9)
print(domain_boundary_rho)
def pressure_constraint_fun(params):
    # function to fix dp/dr=0 at axis and p=0 at edge
    # can modify this for other BC (eg fix p at rho=0.8)
    p_l = params["p_l"]
    dp0 = desc_pressure(Grid(jnp.zeros((1, 3)), jitable=True), p_l, dr=1)
    p1 = desc_pressure(Grid(jnp.zeros((1, 3)).at[0, 0].set(domain_boundary_rho), jitable=True), p_l)
    return jnp.array([dp0, p1]).squeeze()



pressure_constraint_target = jnp.array([0.0, 0.0])

# initial optimization just to get self consistent pressure with fixed initial boundary
pressure_error_weight = jnp.full(yancc_desc_grid.num_rho, 0.01)
stored_energy_weight = 0
objective_from_user_weight = jnp.append(pressure_error_weight, stored_energy_weight)

objectives = [
    ObjectiveFromUser(
        objective_from_user_fun,
        eq,
        target=0,
        weight=objective_from_user_weight,
        grid=yancc_desc_grid,
        deriv_mode="fwd", 
        use_jit=False,# need this assuming manta only has vjp, if using jvp switch to fwd
    )
]

constraints = [
    ForceBalance(eq=eq),  # J x B - grad(p) = 0
    FixCurrent(eq=eq),  # fix zero current, eventually should use real bootstrap
    FixPsi(eq=eq),  # fix total magnetic flux
    FixBoundaryR(eq=eq),  # fix boundary shape
    FixBoundaryZ(eq=eq),
    LinearObjectiveFromUser(
        pressure_constraint_fun, eq, target=pressure_constraint_target
    ),
]

objective = ObjectiveFunction(objectives)
# jax.config.update("jax_log_compiles", True)
# jax.config.update("jax_explain_cache_misses", True)
eq, info_out = eq.optimize(
    objective=objective,
    constraints=constraints,
    optimizer="proximal-lsq-exact",
    maxiter=50,
    verbose=3,
    ftol=0.01,
    x_scale="ess",
    copy=True,
    options={
        # pressure is O(1e4) so we use a larger trust region for the self consistency part
        "initial_trust_radius": 1000.0,
        "max_trust_radius": 1e5,
    },
)
# save for later
eq_self_consistent_pressure = eq.copy()
eq_self_consistent_pressure.save("eq_self_consistent_pressure.h5")

In [ ]:
from desc.plotting import plot_comparison

plot_comparison(
    eqs=[eq_init, eq], labels=["Initial", "self-consistent"]
);

In [ ]:
solver_config = {
    "OutputFilename": "stellarator_opt",
    "Polynomial_degree": 3,
    "Grid_size": 4,
    "tau": 100.0, 
    "Lower_boundary": 0.0,
    "Upper_boundary": 0.9,
    "Relative_tolerance": 0.01,
    "delta_t": 1e-5,
    "MinStepSize": 1e-8,
    "restart": True,
    "solveAdjoint": True, 
}

config = {
    "Stellarator": st_config,
    "Solver": solver_config,
}

manta_objective = make_objective(config, vectorized=True)

# def manta_yancc_fun(fields, grid, Vprime):

#     stored_energy, pressure = Objective(fields, grid, Vprime) 

#     return stored_energy, pressure

def objective_from_user_fun(grid, data):
  # note: don't change the signature to this function
    yancc_dat = {
        "B_sup_t": data["B^theta"],
        "B_sup_z": data["B^zeta"],
        "B_sub_t": data["B_theta"],
        "B_sub_z": data["B_zeta"],
        "Bmag": data["|B|"],
        "dBdt": data["|B|_t"],
        "dBdz": data["|B|_z"],
        "sqrtg": data["sqrt(g)"],
    }

    yancc_dat = {
        key: grid.meshgrid_reshape(val, "rtz") for key, val in yancc_dat.items()
    }

    yancc_dat["Psi"] = grid.compress(
        data["Psi"] / grid.nodes[:, 0] ** 2, surface_label="rho"
    )
    yancc_dat["a_minor"] = jnp.full(grid.num_rho, data["a"])
    yancc_dat["R_major"] = jnp.full(grid.num_rho, data["R0"])
    yancc_dat["iota"] = grid.compress(data["iota"], surface_label="rho")
    yancc_dat["rho"] = grid.compress(grid.nodes[:, 0], surface_label="rho")

    V = grid.compress(data['V(r)'])
    V_r = grid.compress(data['V_r(r)'])
    Vprime = V_r/V[-1]

    fields = jax.vmap(lambda d: yancc.field.Field(**d, NFP=grid.NFP))(yancc_dat)

    desc_pressure = grid.compress(data["p"], surface_label="rho")
    
    stored_energy, manta_pressure = manta_objective(fields, grid, Vprime)
    
    
    pressure_error = manta_pressure - desc_pressure

    # optimization is easiest for least squares objectives, so instead of maximizing
    # stored energy we minimize 1/stored_energy^2 (the squaring happens later)
    return jnp.append(pressure_error, 1 / stored_energy)


desc_data = eq.compute(["V(r)", "V_r(r)"], grid=yancc_desc_grid)

V = yancc_desc_grid.compress(desc_data['V(r)'])
Vn = V/V[-1] # normalize
Vn = interpax.CubicSpline(yancc_rho, Vn)

rho_from_normalized_volume = lambda Vnorm : desc.backend.root_scalar(lambda x: Vn(x) - Vnorm, jnp.sqrt(Vnorm))   



domain_boundary_rho = rho_from_normalized_volume(0.9)
print(domain_boundary_rho)
def pressure_constraint_fun(params):
    # function to fix dp/dr=0 at axis and p=0 at edge
    # can modify this for other BC (eg fix p at rho=0.8)
    p_l = params["p_l"]
    dp0 = desc_pressure(Grid(jnp.zeros((1, 3)), jitable=True), p_l, dr=1)
    p1 = desc_pressure(Grid(jnp.zeros((1, 3)).at[0, 0].set(domain_boundary_rho), jitable=True), p_l)
    return jnp.array([dp0, p1]).squeeze()



pressure_constraint_target = jnp.array([0.0, st.getPressure([0.9])[0]])

In [ ]:
# NOTES:
# Use short time step for self consistent pressure
# Possibly use a smaller weight - definitely too large
# Redo config with a larger timestep for main optimization
# Probably lower teh tolerance
# SUNDIALS notes
# Should adjust the initial timestepping 
# Possibly do one time step and adjust initial timestep based on that
# or just use calc ic and use the post calc ic residual norm

# main optimization, varying boundary to maximize stored energy


# other objectives are non-dimensionalized, so weights should account for that
# and handle relative weighting, this will likely need trial and error
pressure_error_weight = jnp.full(yancc_desc_grid.num_rho, 0.1)
stored_energy_weight = 100.0
objective_from_user_weight = jnp.append(pressure_error_weight, stored_energy_weight)

objectives = [
    AspectRatio(eq=eq, target=6, weight=10),
    Volume(eq=eq, target=V0, weight=10),
    RotationalTransform(eq=eq, target=0.42, weight=10),
    ObjectiveFromUser(
        objective_from_user_fun,
        eq,
        target=0,
        weight=objective_from_user_weight,
        grid=yancc_desc_grid,
        deriv_mode="fwd", 
        use_jit=False,# need this assuming manta only has vjp, if using jvp switch to fwd
    ),
]
constraints = [
    ForceBalance(eq=eq),  # J x B - grad(p) = 0
    FixCurrent(eq=eq),  # fix zero current, eventually should use real bootstrap
    FixPsi(eq=eq),  # fix total magnetic flux
    LinearObjectiveFromUser(
        pressure_constraint_fun, eq, target=pressure_constraint_target
    ),
]

objective = ObjectiveFunction(objectives)

eq, info_out = eq.optimize(
    objective=objective,
    constraints=constraints,
    optimizer="proximal-lsq-exact",
    maxiter=50,
    ftol=0.05,
    verbose=3,
    x_scale="ess",
    options={
        "initial_trust_radius": 0.01,
    },
    copy=True,
)

eq_optimized = eq.copy()


# do a final pass with just the self consistency part to make sure profiles match
pressure_error_weight = jnp.full(yancc_desc_grid.num_rho, 1.0)
stored_energy_weight = 0
objective_from_user_weight = jnp.append(pressure_error_weight, stored_energy_weight)

objectives = [
    ObjectiveFromUser(
        objective_from_user_fun,
        eq,
        target=0,
        weight=objective_from_user_weight,
        grid=yancc_desc_grid,
        deriv_mode="fwd", 
        use_jit=False,# need this assuming manta only has vjp, if using jvp switch to fwd
    )
]

constraints = [
    ForceBalance(eq=eq),  # J x B - grad(p) = 0
    FixCurrent(eq=eq),  # fix zero current, eventually should use real bootstrap
    FixPsi(eq=eq),  # fix total magnetic flux
    FixBoundaryR(eq=eq),  # fix boundary shape
    FixBoundaryZ(eq=eq),
    LinearObjectiveFromUser(
        pressure_constraint_fun, eq, target=pressure_constraint_target
    ),
]

objective = ObjectiveFunction(objectives)

eq, info_out = eq.optimize(
    objective=objective,
    constraints=constraints,
    optimizer="proximal-lsq-exact",
    maxiter=50,
    verbose=3,
    x_scale="ess",
    copy=True,
    options={
        "initial_trust_radius": 1e3,
        "max_trust_radius": 1e5,
    },
)
eq_optimized_self_consistent = eq.copy()

In [ ]:
eq_optimized_self_consistent.save("optimized_equilibrium.h5")

In [ ]:
from desc.plotting import plot_comparison

plot_comparison(
    eqs=[eq_init, eq], labels=["Initial", "self-consistent"]
);

In [ ]:
from desc.plotting import plot_boundaries

plot_comparison(
    eqs=[eq_init, eq_optimized, eq], labels=["Initial", "optimized", "self-consistent"]
);

In [ ]:
plot_boundaries(
    eqs=[eq_init, eq_optimized, eq], labels=["Initial", "optimized", "self-consistent"]
);